<a href="https://colab.research.google.com/github/FernandoJavierNegro/Prueba-2026/blob/main/PRUEBA2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# EfficientDet-Lite1 para detectar botellas

Este cuaderno prepara un dataset YOLO en `.zip`, valida im?genes y etiquetas `.txt`, divide los datos en entrenamiento, validaci?n y prueba, entrena EfficientDet-Lite1 con TensorFlow Lite Model Maker y descarga un paquete listo para Android.

Importante: TensorFlow Lite Model Maker no funciona bien instalado directamente sobre el Python actual de Colab. Por eso este notebook crea un entorno Python 3.11.3 separado y ejecuta el entrenamiento como script dentro de ese entorno compatible.

Active GPU antes de empezar: `Entorno de ejecuci?n > Cambiar tipo de entorno de ejecuci?n > GPU`.

Formato esperado del ZIP:

```text
dataset/
??? images/
?   ??? foto1.jpg
?   ??? foto2.jpg
??? labels/
?   ??? foto1.txt
?   ??? foto2.txt
??? data.yaml
```

Tambi?n admite subcarpetas como `images/train`, `images/val`, `labels/train` y `labels/val`.

Cada `.txt` debe usar formato YOLO:

```text
class_id x_center y_center width height
```

Para una sola clase de botellas:

```text
0 0.5321 0.4812 0.2843 0.6115
```


## Celda 1 ? Crear entorno compatible

Ejecute esta celda una sola vez por sesi?n de Colab. Puede tardar varios minutos.


In [ ]:
%%bash
set -e

# ============================================================
# CREAR ENTORNO COMPATIBLE PARA TFLITE MODEL MAKER
# ============================================================

CONDA_DIR="/content/miniconda"
ENV_NAME="tflite_mm311"

if [ ! -d "$CONDA_DIR" ]; then
    echo "Instalando Miniconda con Python 3.11.3..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-py39_23.3.1-0-Linux-x86_64.sh -O /content/miniconda.sh
    bash /content/miniconda.sh -b -f -p "$CONDA_DIR"
fi

source "$CONDA_DIR/etc/profile.d/conda.sh"

if ! conda env list | awk '{print $1}' | grep -qx "$ENV_NAME"; then
    echo "Creando entorno $ENV_NAME..."
    conda create -y -n "$ENV_NAME" python=3.11.3
fi

conda activate "$ENV_NAME"

python -m pip install -q --upgrade pip==23.3.2 setuptools wheel
python -m pip install -q numpy==1.23.4 protobuf==3.20.3
python -m pip install -q pycocotools scikit-learn pandas pillow pyyaml
python -m pip install -q tflite-model-maker==0.4.3

python - <<'PY'
from tflite_model_maker import model_spec, object_detector
from tflite_model_maker.config import QuantizationConfig
print("? TensorFlow Lite Model Maker funciona en el entorno Python 3.11.3.")
PY

echo "? Celda 1 finalizada. Contin?e con la celda 2."


## Celda 2 ? Subir el ZIP del dataset


In [ ]:
# ============================================================
# SUBIR DATASET YOLO EN ZIP
# ============================================================

from pathlib import Path
from google.colab import files
import shutil

print("Seleccione el archivo ZIP que contiene im?genes y etiquetas YOLO .txt")
archivos_subidos = files.upload()

if not archivos_subidos:
    raise RuntimeError("No se seleccion? ning?n archivo.")

ruta_zip = None
for nombre_archivo in archivos_subidos:
    if nombre_archivo.lower().endswith(".zip"):
        ruta_zip = Path("/content") / nombre_archivo
        break

if ruta_zip is None:
    raise ValueError("Debe seleccionar un archivo .zip")

zip_estandar = Path("/content/dataset_botellas.zip")
if zip_estandar.exists():
    zip_estandar.unlink()

shutil.move(str(ruta_zip), str(zip_estandar))
print(f"? Dataset cargado: {zip_estandar}")


## Celda 3 ? Crear script de entrenamiento


In [ ]:
%%writefile /content/train_efficientdet_lite1.py
# ============================================================
# ENTRENAMIENTO EFFICIENTDET-LITE1 PARA ANDROID
# ============================================================

import argparse
import csv
import gc
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import yaml

from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split

from tflite_model_maker import model_spec
from tflite_model_maker import object_detector
from tflite_model_maker.config import QuantizationConfig


SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)

NOMBRES_CLASES = {0: "botellas"}
PORCENTAJE_TRAIN = 0.70
PORCENTAJE_VALIDATION = 0.20
PORCENTAJE_TEST = 0.10
EPOCHS = 60
BATCH_SIZE = 8
TRAIN_WHOLE_MODEL = True

BASE_DIR = Path("/content/efficientdet_lite1_botellas")
DATASET_EXTRAIDO = BASE_DIR / "dataset_extraido"
DATASET_PREPARADO = BASE_DIR / "dataset_preparado"
IMAGES_DIR = DATASET_PREPARADO / "images"
EXPORT_DIR = BASE_DIR / "modelo_exportado"
RESULTADOS_DIR = BASE_DIR / "resultados"
CSV_MODELO = DATASET_PREPARADO / "annotations_model_maker.csv"
CSV_LECTURA = DATASET_PREPARADO / "annotations_legible.csv"
MODELO_FP16 = EXPORT_DIR / "efficientdet_lite1_botellas_fp16.tflite"
MODELO_INT8 = EXPORT_DIR / "efficientdet_lite1_botellas_int8.tflite"
PAQUETE_ANDROID = BASE_DIR / "paquete_android"
ZIP_FINAL = Path("/content/efficientdet_lite1_botellas_android.zip")
EXTENSIONES_IMAGEN = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".jfif"}


def limpiar_carpetas():
    if BASE_DIR.exists():
        shutil.rmtree(BASE_DIR)
    if ZIP_FINAL.exists():
        ZIP_FINAL.unlink()
    for carpeta in [DATASET_EXTRAIDO, DATASET_PREPARADO, IMAGES_DIR, EXPORT_DIR, RESULTADOS_DIR, PAQUETE_ANDROID]:
        carpeta.mkdir(parents=True, exist_ok=True)


def extraer_zip(ruta_zip):
    try:
        with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:
            archivo_zip.extractall(DATASET_EXTRAIDO)
    except zipfile.BadZipFile as exc:
        raise ValueError("El archivo seleccionado no es un ZIP v?lido.") from exc


def buscar_archivos():
    imagenes = sorted(
        ruta for ruta in DATASET_EXTRAIDO.rglob("*")
        if ruta.is_file() and ruta.suffix.lower() in EXTENSIONES_IMAGEN
    )
    etiquetas = sorted(ruta for ruta in DATASET_EXTRAIDO.rglob("*.txt") if ruta.is_file())
    if not imagenes:
        raise RuntimeError("No se encontraron im?genes dentro del ZIP.")
    if not etiquetas:
        raise RuntimeError("No se encontraron archivos TXT con etiquetas YOLO.")
    return imagenes, etiquetas


def crear_indice_etiquetas(etiquetas):
    indice = {}
    for etiqueta in etiquetas:
        indice.setdefault(etiqueta.stem.lower(), []).append(etiqueta)
    return indice


def buscar_label_correspondiente(ruta_imagen, indice_etiquetas):
    partes = list(ruta_imagen.parts)
    partes_minusculas = [parte.lower() for parte in partes]

    if "images" in partes_minusculas:
        posicion = partes_minusculas.index("images")
        partes_label = partes.copy()
        partes_label[posicion] = "labels"
        candidato = Path(*partes_label).with_suffix(".txt")
        if candidato.exists():
            return candidato

    candidato = ruta_imagen.with_suffix(".txt")
    if candidato.exists():
        return candidato

    candidatos = indice_etiquetas.get(ruta_imagen.stem.lower(), [])
    return candidatos[0] if candidatos else None


def leer_label_yolo(ruta_label):
    cajas = []
    errores = []
    try:
        lineas = ruta_label.read_text(encoding="utf-8", errors="ignore").splitlines()
    except OSError as exc:
        return cajas, [f"No se pudo leer el TXT: {exc}"]

    for numero_linea, linea in enumerate(lineas, start=1):
        linea = linea.strip()
        if not linea:
            continue
        partes = linea.split()
        if len(partes) < 5:
            errores.append(f"L?nea {numero_linea}: formato incompleto")
            continue
        try:
            class_id = int(float(partes[0]))
            x_center = float(partes[1])
            y_center = float(partes[2])
            box_width = float(partes[3])
            box_height = float(partes[4])
        except ValueError:
            errores.append(f"L?nea {numero_linea}: valores no num?ricos")
            continue

        if class_id not in NOMBRES_CLASES:
            errores.append(f"L?nea {numero_linea}: clase {class_id} no configurada")
            continue
        if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < box_width <= 1 and 0 < box_height <= 1):
            errores.append(f"L?nea {numero_linea}: caja fuera del rango normalizado")
            continue

        xmin = max(0.0, x_center - box_width / 2)
        ymin = max(0.0, y_center - box_height / 2)
        xmax = min(1.0, x_center + box_width / 2)
        ymax = min(1.0, y_center + box_height / 2)
        if xmax <= xmin or ymax <= ymin:
            errores.append(f"L?nea {numero_linea}: caja sin superficie")
            continue

        cajas.append({
            "class_id": class_id,
            "class_name": NOMBRES_CLASES[class_id],
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
        })
    return cajas, errores


def validar_dataset(imagenes, indice_etiquetas):
    registros = []
    errores_dataset = []
    imagenes_sin_txt = 0
    imagenes_sin_cajas = 0
    imagenes_invalidas = 0

    for ruta_imagen in imagenes:
        ruta_label = buscar_label_correspondiente(ruta_imagen, indice_etiquetas)
        if ruta_label is None:
            imagenes_sin_txt += 1
            errores_dataset.append({"imagen": str(ruta_imagen), "error": "No se encontr? TXT correspondiente"})
            continue

        cajas, errores_label = leer_label_yolo(ruta_label)
        for error in errores_label:
            errores_dataset.append({"imagen": str(ruta_imagen), "label": str(ruta_label), "error": error})
        if not cajas:
            imagenes_sin_cajas += 1
            continue

        try:
            with Image.open(ruta_imagen) as imagen:
                imagen.verify()
            with Image.open(ruta_imagen) as imagen:
                ancho, alto = imagen.size
            if ancho <= 0 or alto <= 0:
                raise ValueError("Dimensiones no v?lidas")
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            imagenes_invalidas += 1
            errores_dataset.append({"imagen": str(ruta_imagen), "error": str(exc)})
            continue

        registros.append({
            "image_path": str(ruta_imagen),
            "label_path": str(ruta_label),
            "image_width": ancho,
            "image_height": alto,
            "boxes": cajas,
        })

    df = pd.DataFrame(registros)
    print("=" * 60)
    print("VALIDACI?N DEL DATASET")
    print("=" * 60)
    print(f"Im?genes v?lidas:           {len(df)}")
    print(f"Im?genes sin TXT:           {imagenes_sin_txt}")
    print(f"Im?genes sin cajas v?lidas: {imagenes_sin_cajas}")
    print(f"Im?genes inv?lidas:         {imagenes_invalidas}")

    if errores_dataset:
        pd.DataFrame(errores_dataset).to_csv(RESULTADOS_DIR / "errores_dataset.csv", index=False, encoding="utf-8-sig")
    if len(df) < 10:
        raise RuntimeError("Se requieren al menos 10 im?genes v?lidas para dividir train, validation y test.")
    return df


def dividir_dataset(df):
    indices = np.arange(len(df))
    indices_train, indices_temporales = train_test_split(
        indices,
        test_size=PORCENTAJE_VALIDATION + PORCENTAJE_TEST,
        random_state=SEMILLA,
        shuffle=True,
    )
    proporcion_test_temporal = PORCENTAJE_TEST / (PORCENTAJE_VALIDATION + PORCENTAJE_TEST)
    indices_validation, indices_test = train_test_split(
        indices_temporales,
        test_size=proporcion_test_temporal,
        random_state=SEMILLA,
        shuffle=True,
    )
    df_train = df.iloc[indices_train].reset_index(drop=True)
    df_validation = df.iloc[indices_validation].reset_index(drop=True)
    df_test = df.iloc[indices_test].reset_index(drop=True)
    print(f"Train:      {len(df_train)} im?genes")
    print(f"Validation: {len(df_validation)} im?genes")
    print(f"Test:       {len(df_test)} im?genes")
    return df_train, df_validation, df_test


def crear_csv_model_maker(df_train, df_validation, df_test):
    filas_model_maker = []
    filas_legibles = []
    nombres_usados = set()

    def nombre_unico(ruta_imagen, contador):
        nombre = ruta_imagen.name
        if nombre.lower() not in nombres_usados:
            nombres_usados.add(nombre.lower())
            return nombre
        nombre = f"{contador:06d}_{ruta_imagen.stem}{ruta_imagen.suffix.lower()}"
        nombres_usados.add(nombre.lower())
        return nombre

    def agregar_split(dataframe, split):
        for contador, (_, fila) in enumerate(dataframe.iterrows(), start=1):
            ruta_original = Path(fila["image_path"])
            nombre_salida = nombre_unico(ruta_original, contador)
            ruta_salida = IMAGES_DIR / nombre_salida
            shutil.copy2(ruta_original, ruta_salida)
            ruta_absoluta = str(ruta_salida.resolve())
            for numero_box, caja in enumerate(fila["boxes"], start=1):
                filas_model_maker.append([
                    split,
                    ruta_absoluta,
                    caja["class_name"],
                    caja["xmin"],
                    caja["ymin"],
                    "",
                    "",
                    caja["xmax"],
                    caja["ymax"],
                    "",
                    "",
                ])
                filas_legibles.append({
                    "split": split,
                    "filename": nombre_salida,
                    "image_path": ruta_absoluta,
                    "image_width": fila["image_width"],
                    "image_height": fila["image_height"],
                    "bounding_box_number": numero_box,
                    "class_id": caja["class_id"],
                    "class_name": caja["class_name"],
                    "xmin_normalized": caja["xmin"],
                    "ymin_normalized": caja["ymin"],
                    "xmax_normalized": caja["xmax"],
                    "ymax_normalized": caja["ymax"],
                })

    agregar_split(df_train, "TRAINING")
    agregar_split(df_validation, "VALIDATION")
    agregar_split(df_test, "TEST")

    with open(CSV_MODELO, "w", newline="", encoding="utf-8") as archivo_csv:
        csv.writer(archivo_csv).writerows(filas_model_maker)
    pd.DataFrame(filas_legibles).to_csv(CSV_LECTURA, index=False, encoding="utf-8-sig")

    with open(DATASET_PREPARADO / "data.yaml", "w", encoding="utf-8") as archivo_yaml:
        yaml.safe_dump({"nc": len(NOMBRES_CLASES), "names": NOMBRES_CLASES}, archivo_yaml, allow_unicode=True, sort_keys=False)
    with open(DATASET_PREPARADO / "labels.txt", "w", encoding="utf-8") as archivo_labels:
        for class_id in sorted(NOMBRES_CLASES):
            archivo_labels.write(NOMBRES_CLASES[class_id] + "\n")

    print(f"CSV para Model Maker: {CSV_MODELO}")
    print(f"Bounding boxes totales: {len(filas_model_maker)}")
    return len(filas_model_maker)


def entrenar_y_exportar(df_train, df_validation, df_test, total_boxes):
    train_data, validation_data, test_data = object_detector.DataLoader.from_csv(str(CSV_MODELO))
    if train_data.size == 0 or validation_data.size == 0 or test_data.size == 0:
        raise RuntimeError("Alg?n split qued? vac?o. Revise el tama?o del dataset.")

    spec = model_spec.get("efficientdet_lite1")
    gc.collect()
    modelo = object_detector.create(
        train_data=train_data,
        model_spec=spec,
        validation_data=validation_data,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        train_whole_model=TRAIN_WHOLE_MODEL,
    )

    resultados_tensorflow = modelo.evaluate(test_data)
    (RESULTADOS_DIR / "evaluacion_tensorflow.txt").write_text(str(resultados_tensorflow), encoding="utf-8")

    configuracion_fp16 = QuantizationConfig.for_float16()
    modelo.export(export_dir=str(EXPORT_DIR), tflite_filename=MODELO_FP16.name, quantization_config=configuracion_fp16)
    if not MODELO_FP16.exists():
        raise RuntimeError("No se gener? el modelo TFLite Float16.")

    modelo.export(export_dir=str(EXPORT_DIR), tflite_filename=MODELO_INT8.name)

    resultados_tflite = modelo.evaluate_tflite(str(MODELO_FP16), test_data)
    (RESULTADOS_DIR / "evaluacion_tflite_fp16.txt").write_text(str(resultados_tflite), encoding="utf-8")

    return resultados_tensorflow, resultados_tflite


def preparar_paquete(df_train, df_validation, df_test, total_boxes):
    if PAQUETE_ANDROID.exists():
        shutil.rmtree(PAQUETE_ANDROID)
    PAQUETE_ANDROID.mkdir(parents=True, exist_ok=True)

    for archivo in [
        MODELO_FP16,
        MODELO_INT8,
        DATASET_PREPARADO / "labels.txt",
        DATASET_PREPARADO / "data.yaml",
        CSV_LECTURA,
        RESULTADOS_DIR / "evaluacion_tensorflow.txt",
        RESULTADOS_DIR / "evaluacion_tflite_fp16.txt",
    ]:
        if archivo.exists():
            shutil.copy2(archivo, PAQUETE_ANDROID / archivo.name)

    informacion = {
        "architecture": "EfficientDet-Lite1",
        "framework": "TensorFlow Lite Model Maker",
        "classes": NOMBRES_CLASES,
        "float16_model": MODELO_FP16.name,
        "int8_model": MODELO_INT8.name if MODELO_INT8.exists() else None,
        "train_images": len(df_train),
        "validation_images": len(df_validation),
        "test_images": len(df_test),
        "bounding_boxes": total_boxes,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "android_api": "TensorFlow Lite Task Library ObjectDetector",
    }
    (PAQUETE_ANDROID / "modelo_info.json").write_text(json.dumps(informacion, indent=4, ensure_ascii=False), encoding="utf-8")
    (PAQUETE_ANDROID / "README.txt").write_text(
        """
MODELO EFFICIENTDET-LITE1 PARA ANDROID
======================================

Clase:
- botellas

Uso recomendado:
- Float16: GPU Delegate
- INT8: CPU

Copiar el archivo .tflite elegido a:
app/src/main/assets/

Usar con TensorFlow Lite Task Library ObjectDetector.
        """.strip(),
        encoding="utf-8",
    )

    shutil.make_archive(base_name=str(ZIP_FINAL.with_suffix("")), format="zip", root_dir=PAQUETE_ANDROID)
    print("=" * 65)
    print("PROCESO FINALIZADO")
    print("=" * 65)
    print(f"ZIP final: {ZIP_FINAL}")
    print(f"Tama?o: {ZIP_FINAL.stat().st_size / 1024 / 1024:.2f} MB")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--zip", required=True, help="Ruta al ZIP con dataset YOLO")
    args = parser.parse_args()
    ruta_zip = Path(args.zip)
    if not ruta_zip.exists():
        raise FileNotFoundError(f"No existe el ZIP: {ruta_zip}")

    limpiar_carpetas()
    extraer_zip(ruta_zip)
    imagenes, etiquetas = buscar_archivos()
    indice_etiquetas = crear_indice_etiquetas(etiquetas)
    df = validar_dataset(imagenes, indice_etiquetas)
    df_train, df_validation, df_test = dividir_dataset(df)
    total_boxes = crear_csv_model_maker(df_train, df_validation, df_test)
    entrenar_y_exportar(df_train, df_validation, df_test, total_boxes)
    preparar_paquete(df_train, df_validation, df_test, total_boxes)


if __name__ == "__main__":
    main()


## Celda 4 ? Entrenar y exportar modelos

Esta celda ejecuta el entrenamiento dentro del entorno Python 3.11.3. Al finalizar genera `/content/efficientdet_lite1_botellas_android.zip`.


In [ ]:
%%bash
set -e

# ============================================================
# EJECUTAR ENTRENAMIENTO EN EL ENTORNO PYTHON 3.9
# ============================================================

source /content/miniconda/etc/profile.d/conda.sh
conda activate tflite_mm311

python /content/train_efficientdet_lite1.py --zip /content/dataset_botellas.zip


## Celda 5 ? Descargar resultado


In [ ]:
# ============================================================
# DESCARGAR PAQUETE FINAL
# ============================================================

from pathlib import Path
from google.colab import files

zip_final = Path("/content/efficientdet_lite1_botellas_android.zip")

if not zip_final.exists():
    raise FileNotFoundError("Todav?a no existe el ZIP final. Ejecute primero la celda de entrenamiento.")

print(f"Descargando: {zip_final.name}")
files.download(str(zip_final))


## Uso b?sico en Android

Coloque el modelo elegido en:

```text
app/src/main/assets/efficientdet_lite1_botellas_fp16.tflite
```

Dependencias sugeridas:

```gradle
dependencies {
    implementation "org.tensorflow:tensorflow-lite-task-vision"
    implementation "org.tensorflow:tensorflow-lite-gpu-delegate-plugin"
}
```

Inicializaci?n en Kotlin:

```kotlin
import org.tensorflow.lite.task.core.BaseOptions
import org.tensorflow.lite.task.vision.detector.ObjectDetector

val baseOptions = BaseOptions.builder()
    .useGpu()
    .build()

val options = ObjectDetector.ObjectDetectorOptions.builder()
    .setBaseOptions(baseOptions)
    .setScoreThreshold(0.35f)
    .setMaxResults(20)
    .build()

val detector = ObjectDetector.createFromFileAndOptions(
    context,
    "efficientdet_lite1_botellas_fp16.tflite",
    options
)
```

Si el GPU Delegate da problemas en un tel?fono, pruebe el modelo INT8 y quite `.useGpu()`.
